# Plate 06 · Export (ONNX · LiteRT · Core ML)

Produces the **base plate package** `Weights/plate/<version>/`:
`plate_detector.onnx/.tflite/.mlpackage`, `plate_recognizer.onnx/.tflite/.mlpackage`, `Model_Card.json`.
The backend signs + encrypts packages per tenant before the apps download them (spec section 11.3).

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Plate_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import torch
from ultralytics import YOLO
from Common_Utils.Export_Utils import export_onnx, export_litert, export_coreml, write_model_card
from Common_Utils.Kuwait_Plate_Layout import CELL_W, CELL_H

VERSION = "1.0.0-kw"
OUT = model_dir("plate", VERSION)
best = RUNS / "Plate_Detector" / "yolo11n_pose_kw_v1" / "weights" / "best.pt"
det = YOLO(str(best))
det.export(format="onnx", imgsz=640, simplify=True)
det.export(format="tflite", imgsz=640, int8=True, data=str(DATA / "Plates" / "Detection" / "plates.yaml"))
# det.export(format="coreml", imgsz=640)   # run on macOS

In [ ]:
# Cell recogniser (re-create the CRNN class from notebook 04 or import it from a .py module)
# crnn = CRNN(); crnn.load_state_dict(torch.load(RUNS / "crnn_last.pt", map_location="cpu"))
sample = torch.zeros(1, 1, CELL_H, CELL_W)
# export_onnx(crnn, sample, OUT / "plate_recognizer.onnx")
# export_litert(crnn, sample, OUT / "plate_recognizer.tflite", int8=True)
# export_coreml(crnn, sample, OUT / "plate_recognizer.mlpackage")   # macOS

In [ ]:
write_model_card(OUT, kind="plate", version=VERSION,
    metrics={"det_map50": None, "corner_error_pct": None, "exact_match_day": None, "exact_match_night": None, "row_agreement": None},
    data_sources=["Kuwaiti pilot-station capture (consented)", "Open Images v7 plates (CC-BY 4.0)", "Synthetic Kuwaiti plates"],
    input_spec={"detector": "1x3x640x640 RGB 0-1 -> boxes + 4 corners", "recognizer": f"1x1x{CELL_H}x{CELL_W} gray cell, (x-0.5)/0.5",
                "layout": "Common_Utils/Kuwait_Plate_Layout.py (canonical 520x120, 4 cells)"},
    notes="Base model for Kuwaiti plates - contains no station data.")